In [1]:
# ===============================================
# IMAGE CLASSIFICATION PIPELINE (AUTO LABELING)
# Menggunakan Feature Extraction + Clustering
# ===============================================

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# ===============================================
# 1. Directory Configuration (SESUAIKAN DENGAN STRUKTUR KAMU)
# ===============================================
base_dir = os.path.join(os.getcwd(), "dataset")  # otomatis menunjuk ke folder 'dataset'
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

# ===============================================
# 2. Load Data dan Ekstrak Fitur
# ===============================================
feature_extractor = EfficientNetB0(weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))

def extract_features(directory):
    features = []
    filenames = []
    for file in tqdm(os.listdir(directory)):
        file_path = os.path.join(directory, file)
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        try:
            img = load_img(file_path, target_size=(224, 224))
            img_array = img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)
            feature = feature_extractor.predict(img_array, verbose=0)
            features.append(feature.flatten())
            filenames.append(file)
        except Exception as e:
            print(f"⚠️ Error processing {file}: {e}")
            continue
    return np.array(features), filenames

print("🔍 Ekstraksi fitur train images...")
train_features, train_filenames = extract_features(train_dir)
print("✅ Fitur train:", train_features.shape)

# ===============================================
# 3. Clustering untuk Auto Labeling
# ===============================================
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)

n_clusters = 15  # jumlah kelas
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(train_features_scaled)

unique, counts = np.unique(cluster_labels, return_counts=True)
print("📊 Distribusi label otomatis:", dict(zip(unique, counts)))

auto_labels = pd.DataFrame({
    'filename': train_filenames,
    'label': cluster_labels.astype(str)
})
auto_labels.to_csv(os.path.join(base_dir, "auto_train_labels.csv"), index=False)
print("✅ File auto_train_labels.csv berhasil dibuat di folder 'dataset'.")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
🔍 Ekstraksi fitur train images...


100%|██████████| 4257/4257 [23:34<00:00,  3.01it/s]


✅ Fitur train: (4257, 1280)
📊 Distribusi label otomatis: {0: 191, 1: 354, 2: 295, 3: 378, 4: 283, 5: 200, 6: 447, 7: 386, 8: 433, 9: 155, 10: 248, 11: 178, 12: 68, 13: 413, 14: 228}
✅ File auto_train_labels.csv berhasil dibuat di folder 'dataset'.


In [2]:


# ===============================================
# 4. Train-Validation Split
# ===============================================
train_df, val_df = train_test_split(auto_labels, test_size=0.2, stratify=auto_labels['label'], random_state=42)

# ===============================================
# 5. Data Generator
# ===============================================
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=train_dir,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

# ===============================================
# 6. Model (Transfer Learning)
# ===============================================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Bekukan sebagian layer awal saja
for layer in base_model.layers[:200]:
    layer.trainable = False
for layer in base_model.layers[200:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(n_clusters, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint(os.path.join(base_dir, 'best_model.keras'), monitor='val_accuracy', save_best_only=True)
]

# ===============================================
# 7. Training
# ===============================================
print("🚀 Mulai Training Model...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=callbacks
)

# ===============================================
# 8. Fine-Tuning
# ===============================================
base_model.trainable = True
for layer in base_model.layers[:150]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(os.path.join(base_dir, 'best_model.keras'), monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

print("🔧 Fine-tuning model...")
fine_tune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)

# ===============================================
# 9. Evaluasi
# ===============================================
val_loss, val_acc = model.evaluate(val_generator)
print(f"✅ Validation Accuracy: {val_acc:.4f}")

# ===============================================
# 10. Prediksi Test Set
# ===============================================
test_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_df = pd.DataFrame({'filename': test_files})
test_df['filepath'] = test_df['filename'].apply(lambda x: os.path.join(test_dir, x))

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col=None,
    target_size=(224, 224),
    class_mode=None,
    batch_size=32,
    shuffle=False
)

predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)

# ===============================================
# 11. Submission
# ===============================================
submission = pd.DataFrame({
    'filename': test_df['filename'],
    'label': predicted_classes
})
submission.to_csv(os.path.join(base_dir, 'submission.csv'), index=False)
print("🎯 submission.csv berhasil dibuat di folder 'dataset'!")


Found 3405 validated image filenames belonging to 15 classes.
Found 852 validated image filenames belonging to 15 classes.
🚀 Mulai Training Model...
Epoch 1/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 261s 2s/step - accuracy: 0.0846 - loss: 2.6887 - val_accuracy: 0.1045 - val_loss: 2.6387
Epoch 2/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.0990 - loss: 2.6609 - val_accuracy: 0.0833 - val_loss: 2.6391
Epoch 3/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.1001 - loss: 2.6595 - val_accuracy: 0.1045 - val_loss: 2.6282
Epoch 4/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.1063 - loss: 2.6304 - val_accuracy: 0.0469 - val_loss: 3.5726
Epoch 5/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 225s 2s/step - accuracy: 0.1181 - loss: 2.5867 - val_accuracy: 0.0481 - val_loss: 6.3820
Epoch 6/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 166s 2s/step - accuracy: 0.1304 - loss: 2.5583 - val_accuracy: 0.1620 - val_loss: 2.4479
Epoch 7/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 155s 1s/step - accuracy: 0.1310 - los